# 02 · Preprocesamiento y control de calidad

Se documenta y comprueba el pipeline existente. No se generan nuevas anotaciones ni se sobrescribe el dataset al ejecutar este notebook.

In [1]:
from pathlib import Path
import sys, json, csv, hashlib
from html import escape
from IPython.display import display, Markdown, HTML, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/paths.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
def load(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))
def table(rows):
    if not rows:
        return display(Markdown("Sin filas."))
    keys = list(rows[0])
    head = "".join("<th>" + escape(str(k)) + "</th>" for k in keys)
    body = "".join("<tr>" + "".join("<td>" + escape(str(r.get(k, ""))) + "</td>" for k in keys) + "</tr>" for r in rows)
    display(HTML("<table><thead><tr>"+head+"</tr></thead><tbody>"+body+"</tbody></table>"))
print("Raíz del proyecto localizada. Se utilizan evidencias reales del repositorio.")


Raíz del proyecto localizada. Se utilizan evidencias reales del repositorio.


## Preparación del detector

`src.preparar_dataset` selecciona filas revisadas SI, valida cajas YOLO, agrupa por SHA256 de píxeles y divide grupos con semilla 42. No convierte archivos sin etiqueta en negativos. La auditoría publica sólo estadísticas.

In [2]:
eda = load("results/metrics/analisis_datos.json")
table([{"control":k,"resultado":eda[k]} for k in ["review_rows","review_marked_yes","excluded_rows","issue_count","manifest_group_crossings"]])
from src.preparar_dataset import validar_etiqueta
dataset = ROOT / "data/processed/dataset_placas"
labels = sorted((dataset / "labels").rglob("*.txt")) if dataset.is_dir() else []
if labels:
    for path in labels:
        validar_etiqueta(path)
    print("Etiquetas locales validadas en esta ejecución:",len(labels))
else:
    print("Dataset no incluido en esta copia: se consulta auditoría guardada, sin fingir nueva validación.")

control,resultado
review_rows,13621
review_marked_yes,7332
excluded_rows,0
issue_count,0
manifest_group_crossings,0


Etiquetas locales validadas en esta ejecución: 7332


## Preprocesamiento real de la web

Validación de formato, límite de 12 MiB y 20 megapíxeles, orientación EXIF y RGB. El detector recibe BGR; el OCR vuelve a RGB y reescala a 128×64 con interpolación cúbica. No se implementan CLAHE ni sharpening.

In [3]:
from app.web_service import decode_image
import numpy as np
photo = decode_image((ROOT/"app/static/examples/gca4771.png").read_bytes())
rgb = np.asarray(photo)
bgr = np.ascontiguousarray(rgb[:,:,::-1])
assert np.array_equal(bgr[:,:,::-1],rgb)
print("Dimensiones RGB reales:",rgb.shape,"; tipo:",rgb.dtype,"; conversión BGR reversible comprobada.")

Dimensiones RGB reales: (349, 595, 3) ; tipo: uint8 ; conversión BGR reversible comprobada.


## Normalización de texto y datos faltantes

Se eliminan espacios/guiones y se pasa a mayúsculas; no se sustituyen O/0 o I/1. Las lecturas ilegibles no reciben etiquetas inventadas.

In [4]:
from app.web_validation import declared_plate
table([{"entrada":s,"normalizada":declared_plate(s)} for s in ["gca-4771"," GTW 9311 ","ABO123","AB0123"]])
assert declared_plate("ABO123") != declared_plate("AB0123")

entrada,normalizada
gca-4771,GCA4771
GTW 9311,GTW9311
ABO123,ABO123
AB0123,AB0123


Los valores extremos se revisan, no se eliminan automáticamente. Una geometría válida no certifica la exactitud visual de las anotaciones. Véase `docs/analisis_datos.md`.